In [0]:
%python
# CELDA 1: Lectura de parámetros (Python)
try:
    # Capturar del Deploy de GitHub Actions
    storageName = spark.conf.get("spark.databricks.job.notebook.parameters.storageName")
    catalog = spark.conf.get("spark.databricks.job.notebook.parameters.catalog")
    credential = spark.conf.get("spark.databricks.job.notebook.parameters.credential")
    print("-> Parámetros de producción inyectados desde GitHub Actions con éxito.")
except Exception:
    # Inicializar widgets locales para Desarrollo manual
    print("-> Modo manual (Desarrollo). Inicializando widgets locales...")
    dbutils.widgets.text("storageName", "datalakefforero03")
    dbutils.widgets.text("catalog", "formula1")
    dbutils.widgets.text("credential", "credential_formula1")
    
    storageName = dbutils.widgets.get("storageName")
    catalog = dbutils.widgets.get("catalog")
    credential = dbutils.widgets.get("credential")

print(f"VALOR_CATALOGO: {catalog}")
print(f"VALOR_STORAGE: {storageName}")

# 3. Guardarlos para que la celda SQL los formatee como tú deseas
spark.conf.set("v_catalog", catalog)
spark.conf.set("v_storage", storageName)

In [0]:

# 2. CREACIÓN DE OBJETOS USANDO F-STRINGS DE PYTHON (Sintaxis de llaves)
# =====================================================================
ruta_managed = f"abfss://metastore@{storageName}.dfs.core.windows.net/"
print(f"Ruta de almacenamiento formateada: {ruta_managed}")

# 2. Inyectar de forma limpia en el Query SQL de Spark
query_catalog = f"""
CREATE CATALOG IF NOT EXISTS {catalog}
MANAGED LOCATION '{ruta_managed}'
COMMENT 'Catalogo de produccion de formula 1';
"""

print(f"Ejecutando en Databricks: {query_catalog}")
spark.sql(query_catalog)

# 3. Crear los esquemas en la arquitectura Medallion
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze;")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver;")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold;")

print("¡Proceso finalizado en el cuaderno!")